# Fine-tuning Qwen-1.5-1.8B-Chat Manually with SFTTrainer

This notebook guides you through fine-tuning the `Qwen/Qwen1.5-1.8B-Chat` model using the standard Hugging Face libraries (`transformers`, `datasets`, `peft`, `trl`) on Google Colab.

**Goal:** Fine-tune the model on custom data loaded from Google Drive and save the resulting LoRA adapter weights back to Google Drive.

**Steps:**
1. Setup Colab Environment (GPU, Libraries)
2. Mount Google Drive & Define Paths
3. Authenticate with Hugging Face (Optional)
4. Load and Prepare Data
5. Load Base Model and Tokenizer (with Quantization)
6. Configure PEFT (LoRA)
7. Configure Training Arguments
8. Initialize SFTTrainer
9. Start Fine-tuning
10. Save Final Adapter to Google Drive
11. (Optional) Test the Fine-tuned Model

## 1. Setup Colab Environment

First, ensure you are using a GPU runtime:
* Go to `Runtime` -> `Change runtime type`
* Select `GPU` (T4 is usually available for free) and click `Save`.

In [ ]:
# Install necessary libraries
!pip install -U transformers datasets accelerate peft bitsandbytes trl pandas pyarrow torch -q

## 2. Mount Google Drive & Define Paths

Mount Google Drive to access the input data and specify where to save the final model adapter.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# --- IMPORTANT: Define paths --- 

# Adjust this path based on where you saved the file in your Google Drive
google_drive_base_path = '/content/drive/MyDrive/' # Or '/content/drive/MyDrive/YourFolder/'
parquet_filename = 'finetune_data.parquet'
input_parquet_path = os.path.join(google_drive_base_path, parquet_filename)

# Define the path in Google Drive where the final LoRA adapter will be saved
# Choose a descriptive name
output_model_dir_drive = os.path.join(google_drive_base_path, 'qwen1.5-1.8b-cfe-racfe-adapter')

# Define a temporary directory in Colab for checkpoints during training
# This avoids slow writes to Google Drive during training
output_dir_temporary = '/content/qwen_finetune_results'

print(f"Input data path: {input_parquet_path}")
print(f"Temporary output dir (Colab): {output_dir_temporary}")
print(f"Final model save path (Drive): {output_model_dir_drive}")

# Create the temporary output directory if it doesn't exist
os.makedirs(output_dir_temporary, exist_ok=True)

## 3. Authenticate with Hugging Face (Optional)

Login if you need to access gated models/datasets or if you plan to push the final model to the Hub later (though this notebook saves to Drive).

In [ ]:
from huggingface_hub import notebook_login

# Uncomment and run if needed
# notebook_login()

## 4. Load and Prepare Data

Load the Parquet file and format it into a single `text` column using the ChatML format suitable for Qwen models.

In [ ]:
from datasets import load_dataset
import pandas as pd # Using pandas as intermediate for easier formatting initially
from datasets import Dataset

# Load the Parquet data using pandas first for easier row-wise formatting
try:
    df = pd.read_parquet(input_parquet_path)
    print(f"Loaded {len(df)} records from {input_parquet_path}")
except Exception as e:
    print(f"Error loading Parquet file: {e}\nCheck the 'input_parquet_path' variable.")
    # Stop execution if loading fails
    raise e

# Define the formatting function (using ChatML format for Qwen)
def format_chatml(row):
    instruction = row['instruction']
    input_data = row['input']
    output_data = row['output']
    
    # Combine instruction and input based on the task type
    if "RTL 代码片段" in instruction: # Graph Analysis
         user_content = f"{instruction}\n\nInput Code:\n```\n{input_data}\n```"
    elif "判断代码中存在的保护类型" in instruction: # Protection Determination
         user_content = f"{instruction}\n\nInput Graph JSON:\n```json\n{input_data}\n```"
    elif "验证代码是否符合规则" in instruction: # Rule Verification
         user_content = f"{instruction}\n\nInput Data JSON:\n```json\n{input_data}\n```"
    else:
         user_content = f"{instruction}\n\nInput:\n{input_data}" # Fallback
    
    # Format as ChatML messages dictionary list (SFTTrainer can often handle this directly)
    # If SFTTrainer struggles, we might need to manually apply the tokenizer's chat template later
    messages = [
        {"role": "system", "content": "You are a helpful assistant specialized in RTL code analysis and security verification."},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output_data} # The expected output
    ]
    
    # For SFTTrainer, it's often better to provide the structured messages 
    # or a single 'text' field formatted EXACTLY as the tokenizer expects.
    # Let's try the structured message format first, but keep the text formatting as backup.
    
    # Simple text representation (Backup if structured messages don't work well with SFTTrainer setup)
    text_formatted = f"<|im_start|>system\n{messages[0]['content']}<|im_end|>\n<|im_start|>user\n{messages[1]['content']}<|im_end|>\n<|im_start|>assistant\n{messages[2]['content']}<|im_end|>"
    
    # Return both for now, decide which one to use with SFTTrainer later
    # return pd.Series([messages, text_formatted])
    # Let's stick to the text format for simplicity with SFTTrainer's 'text_dataset' field
    return text_formatted

# Apply the formatting to create the 'text' column
if 'df' in locals():
    df['text'] = df.apply(format_chatml, axis=1)
    
    # Keep only the 'text' column
    df_formatted = df[['text']]
    
    # Convert pandas DataFrame to Hugging Face Dataset
    dataset = Dataset.from_pandas(df_formatted)
    
    print("\nDataset created and formatted.")
    print(dataset)
    print("\nExample formatted text:")
    print(dataset[0]['text'])
else:
    print("DataFrame 'df' not loaded. Cannot prepare data.")

## 5. Load Base Model and Tokenizer (with Quantization)

Load the Qwen-1.5-1.8B-Chat model and its tokenizer. We'll use 4-bit quantization to reduce memory footprint.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# # Define the base model ID
# base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# # Configure quantization (4-bit)
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16 # Use bfloat16 for better performance if available (T4 supports it)
# )

# # Load the base model with quantization
# model = AutoModelForCausalLM.from_pretrained(
#     base_model_id,
#     quantization_config=bnb_config,
#     device_map="auto", # Automatically distribute across available GPUs (or CPU if no GPU)
#     trust_remote_code=True # Needed for some models like Qwen
# )

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the tokenizer
# tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)

# Set padding token if it's not already set
# Qwen tokenizer typically has pad_token set to eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    # Set padding side to right for SFTTrainer to avoid issues
    tokenizer.padding_side = "right" 
    print(f"Set pad_token to {tokenizer.pad_token} and padding_side to {tokenizer.padding_side}")
else:
    print(f"Tokenizer pad token already set: {tokenizer.pad_token}")
    # Ensure padding side is right if using SFTTrainer
    if tokenizer.padding_side != 'right':
        print(f"Setting padding_side to 'right' (was {tokenizer.padding_side})")
        tokenizer.padding_side = 'right'

print("Model and Tokenizer loaded.")

## 6. Configure PEFT (LoRA)

Set up Parameter-Efficient Fine-Tuning using LoRA.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# Prepare the model for k-bit training (important when using quantization)
model.gradient_checkpointing_enable() # Reduces memory usage during training
model = prepare_model_for_kbit_training(model)

# Define LoRA configuration
lora_config = LoraConfig(
    r=16,                  # LoRA rank (dimension)
    lora_alpha=32,         # Alpha scaling factor (often 2*r)
    target_modules="auto", # Automatically find suitable modules (like attention projections) - Recommended for Qwen
    # Example manual target_modules for Qwen (consult model card if 'auto' fails):
    # target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,     # Dropout probability for LoRA layers
    bias="none",          # Whether to train bias parameters (usually 'none' for LoRA)
    task_type="CAUSAL_LM"   # Task type
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print the percentage of trainable parameters
model.print_trainable_parameters()

print("LoRA configured.")

## 7. Configure Training Arguments

Define the hyperparameters for training using `TrainingArguments`.

In [ ]:
from transformers import TrainingArguments

# Define training arguments
# Adapt batch_size and gradient_accumulation_steps based on GPU memory (T4 is limited)
training_args = TrainingArguments(
    output_dir=output_dir_temporary,        # Save checkpoints temporarily in Colab
    per_device_train_batch_size=1,          # Keep low for T4 memory
    gradient_accumulation_steps=8,          # Increase effective batch size (1 * 8 = 8)
    learning_rate=2e-4,                     # Common learning rate for LoRA
    num_train_epochs=1,                     # Start with 1 epoch
    logging_steps=10,                       # Log metrics every 10 steps
    save_strategy="steps",                 # Save based on steps
    save_steps=50,                          # Save a checkpoint every 50 steps (adjust as needed)
    save_total_limit=1,                     # Only keep the latest checkpoint in temp dir
    report_to="none",                   # Disable reporting to WandB/TensorBoard etc.
    fp16=False,                             # Set to False when using 4-bit quantization and bfloat16 compute type
    bf16=True,                              # Use bfloat16 if compute dtype is bfloat16 (recommended for T4+)
    gradient_checkpointing=True,            # Enable gradient checkpointing to save memory
    optim="paged_adamw_8bit",           # Use 8-bit AdamW optimizer to save memory
)

print("Training Arguments configured.")

## 8. Initialize SFTTrainer

Create the `SFTTrainer` instance, which handles the supervised fine-tuning process.

In [ ]:
from trl import SFTTrainer

# Define maximum sequence length
max_seq_length = 1024 # Adjust based on your data and GPU memory

# Initialize the trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset, # Use the formatted dataset
    dataset_text_field="text", # Specify the column containing the formatted text
    tokenizer=tokenizer,
    peft_config=lora_config, # Pass the LoRA configuration
    max_seq_length=max_seq_length,
    # packing=True, # Can pack short sequences together for efficiency (optional)
)

print("SFTTrainer initialized.")

## 9. Start Fine-tuning

Run the training process.

In [ ]:
print("Starting training...")
training_successful = False # Initialize flag
try:
    trainer.train()
    print("Training completed successfully.")
    training_successful = True
except Exception as e:
    print(f"An error occurred during training: {e}")


## 10. Save Final Adapter to Google Drive

After training finishes, save the final LoRA adapter weights to the designated path in your Google Drive.

In [ ]:
if training_successful:
    print(f"Saving final LoRA adapter to: {output_model_dir_drive}")
    try:
        # Ensure the target directory exists in Drive
        os.makedirs(output_model_dir_drive, exist_ok=True)
        
        # Save the adapter weights (and tokenizer config, etc.)
        trainer.save_model(output_model_dir_drive)
        print("Model adapter saved successfully to Google Drive.")
        
        # Optionally save the tokenizer as well if it was modified (though unlikely here)
        # tokenizer.save_pretrained(output_model_dir_drive)
    except Exception as e:
        print(f"Error saving model to Google Drive: {e}")
else:
    print("Skipping saving model to Google Drive due to training errors.")

## 11. (Optional) Test the Fine-tuned Model

Load the base model and the saved adapter from Google Drive to perform inference.

In [ ]:
# Example code to load and test (may require kernel restart for memory)

# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from peft import PeftModel
# 
# # --- Configuration for Loading ---
# base_model_id = "Qwen/Qwen1.5-1.8B-Chat"
# adapter_path = output_model_dir_drive # Path where the adapter was saved in Drive
# 
# print(f"Loading base model: {base_model_id}")
# # Load the base model again (consider using quantization config if used for training)
# bnb_config_inf = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16 
# )
# base_model_for_inference = AutoModelForCausalLM.from_pretrained(
#     base_model_id,
#     quantization_config=bnb_config_inf,
#     device_map="auto",
#     trust_remote_code=True
# )
# 
# tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
# if tokenizer.pad_token is None:
#      tokenizer.pad_token = tokenizer.eos_token
#      tokenizer.padding_side = 'right'
# 
# print(f"Loading LoRA adapter from: {adapter_path}")
# # Load the PeftModel by merging the adapter into the base model
# ft_model = PeftModel.from_pretrained(base_model_for_inference, adapter_path)
# print("Fine-tuned model loaded successfully.")
# 
# # --- Prepare Inference Input ---
# # Prepare your test prompt using the same ChatML format used for training
# test_instruction = "分析提供的 RTL 代码片段，并以 JSON 格式生成一个简化的控制/数据流图。"
# test_input_code = "BB: -2\n... (your test RTL code snippet) ..." # Replace with actual test code
# user_content = f"{test_instruction}\n\nInput Code:\n```\n{test_input_code}\n```"
# 
# messages = [
#     {"role": "system", "content": "You are a helpful assistant specialized in RTL code analysis and security verification."},
#     {"role": "user", "content": user_content}
# ]
# 
# # Apply chat template for inference
# text_prompt = tokenizer.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True # Important for generation tasks
# )
# 
# print("\n--- Input Prompt ---")
# print(text_prompt)
# 
# # --- Run Inference ---
# model_inputs = tokenizer([text_prompt], return_tensors="pt").to(ft_model.device)
# 
# # Generate output
# # Adjust generation parameters as needed (e.g., max_new_tokens, do_sample, temperature)
# generated_ids = ft_model.generate(
#     **model_inputs,
#     max_new_tokens=512 
# )
# 
# # Decode the generated tokens, skipping the prompt part
# response = tokenizer.batch_decode(generated_ids[:, model_inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
# 
# print("\n--- Model Response ---")
# print(response)